In [12]:
from mcs.schemas.dataset import DatasetSpec
from mcs.schemas.split import SplitSpec
from mcs.schemas.embedding import EmbeddingSpec
from mcs.schemas.train import TrainSpec
from mcs.schemas.execution import ExecutionSpec

from mcs.provenance import ProvenanceRecord, ArtifactRef, MetricSummary, LineageGraph
from mcs.registry import LocalRegistry
import yaml

In [5]:
def read_yml(yml_doc):
    with open(yml_doc) as f:
        yml_dict = yaml.safe_load(f)
    return yml_dict

In [6]:
ds_yml = "../examples/dataset.yaml"
sp_yml = "../examples/split.yaml"
em_yml = "../examples/embedding.yaml"
tr_yml = "../examples/train_classification.yaml"
ex_yml = "../examples/execution.yaml"

In [7]:
ds_dict = read_yml(ds_yml)
sp_dict = read_yml(sp_yml)
em_dict = read_yml(em_yml)
tr_dict = read_yml(tr_yml)
ex_dict = read_yml(ex_yml)

In [8]:
dataset = DatasetSpec.model_validate(ds_dict)
split = SplitSpec.model_validate(sp_dict)
embedding = EmbeddingSpec.model_validate(em_dict)
train = TrainSpec.model_validate(tr_dict)
execution = ExecutionSpec.model_validate(ex_dict)

In [9]:
record = ProvenanceRecord.from_specs(
    dataset=dataset,
    split=split,
    embedding=embedding,
    train=train,
    execution=execution,
    artifacts=[
        ArtifactRef(kind="metrics", path="../artifacts/metrics/run.json", format="json"),
        ArtifactRef(kind="predictions", path="../artifacts/predictions/run.parquet", format="parquet"),
    ],
    metric_summary=MetricSummary(split="test", metrics={"roc_auc": 0.91}),
    notes="Case study 1 run."
)

In [10]:
print(record.run_id)
print(record.to_json())

7b4f3d10bb748cd076da3fc0e970b871adb6a47911e8ec6b0261e22497c8064c
{
  "artifacts": [
    {
      "format": "json",
      "kind": "metrics",
      "path": "../artifacts/metrics/run.json"
    },
    {
      "format": "parquet",
      "kind": "predictions",
      "path": "../artifacts/predictions/run.parquet"
    }
  ],
  "created_at_utc": "2026-01-25T11:55:29+00:00",
  "dataset_fingerprint": "d8437ea87fe1941433a8251ad1854c26d4e9b279c8dc49b37402d80772837d6c",
  "dataset_name": "toy_protein_fitness_v1",
  "embedding_fingerprint": "67f616fde142f7c5df0765e96aa792673bd778580e955bbb9df09be094407e89",
  "embedding_name": "esm2_t33_last_mean_fp32_cpu_v1",
  "execution_fingerprint": "541568f06e541f1a0a8a7992eed959ad2293a997a72b5e01424fa79c3e35a09e",
  "execution_name": "dev_linux_cpu_py312_v1",
  "extra": {},
  "mcs_version": "0.1.0",
  "metric_summary": {
    "metrics": {
      "roc_auc": 0.91
    },
    "split": "test"
  },
  "notes": "Case study 1 run.",
  "run_id": "7b4f3d10bb748cd076da3fc0e97

In [11]:
graph = LineageGraph.from_record(record)
print(graph.to_dict())

{'nodes': [{'id': 'dataset:d8437ea87fe1941433a8251ad1854c26d4e9b279c8dc49b37402d80772837d6c', 'type': 'spec', 'label': 'dataset:toy_protein_fitness_v1', 'payload': {'fingerprint': 'd8437ea87fe1941433a8251ad1854c26d4e9b279c8dc49b37402d80772837d6c'}}, {'id': 'split:78229a323ce73e69236771f52c537d716525f15f0b77bc1f48ae637f79d7ca5c', 'type': 'spec', 'label': 'split:cluster_aware_seqid30_v1', 'payload': {'fingerprint': '78229a323ce73e69236771f52c537d716525f15f0b77bc1f48ae637f79d7ca5c'}}, {'id': 'embedding:67f616fde142f7c5df0765e96aa792673bd778580e955bbb9df09be094407e89', 'type': 'spec', 'label': 'embedding:esm2_t33_last_mean_fp32_cpu_v1', 'payload': {'fingerprint': '67f616fde142f7c5df0765e96aa792673bd778580e955bbb9df09be094407e89'}}, {'id': 'train:638392926fb0b4f0e408d2d663ade500840e4235b6d8f9b5c64f1d038d1145af', 'type': 'spec', 'label': 'train:logreg_classification_baseline_v1', 'payload': {'fingerprint': '638392926fb0b4f0e408d2d663ade500840e4235b6d8f9b5c64f1d038d1145af'}}, {'id': 'executio

In [13]:
reg = LocalRegistry(".mcs_registry")
out = reg.register_all(
    dataset=dataset,
    split=split,
    embedding=embedding,
    train=train,
    execution=execution,
    record=record,
    artifact_mode="reference",  # or "copy"
)

In [14]:
print("run_id:", record.run_id)
print(out["run"])
print(out["specs"])

run_id: 7b4f3d10bb748cd076da3fc0e970b871adb6a47911e8ec6b0261e22497c8064c
.mcs_registry/runs/7b4f3d10bb748cd076da3fc0e970b871adb6a47911e8ec6b0261e22497c8064c.json
{'dataset': PosixPath('.mcs_registry/specs/dataset/d8437ea87fe1941433a8251ad1854c26d4e9b279c8dc49b37402d80772837d6c.json'), 'split': PosixPath('.mcs_registry/specs/split/78229a323ce73e69236771f52c537d716525f15f0b77bc1f48ae637f79d7ca5c.json'), 'embedding': PosixPath('.mcs_registry/specs/embedding/67f616fde142f7c5df0765e96aa792673bd778580e955bbb9df09be094407e89.json'), 'train': PosixPath('.mcs_registry/specs/train/638392926fb0b4f0e408d2d663ade500840e4235b6d8f9b5c64f1d038d1145af.json'), 'execution': PosixPath('.mcs_registry/specs/execution/541568f06e541f1a0a8a7992eed959ad2293a997a72b5e01424fa79c3e35a09e.json')}
